In [11]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline

In [12]:
train_path = "../../Dataset/processed/train_data.csv"
test_path = "../../Dataset/processed/test_data.csv"

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

print("Training data loaded successfully")
print("Testing data loaded successfully")

print("Training data shape: ", train_data.shape)
print("Testing dat shape: ", test_data.shape)

Training data loaded successfully
Testing data loaded successfully
Training data shape:  (1356, 2461)
Testing dat shape:  (340, 2461)


In [13]:
X_train = train_data.drop(columns=["Revenue(INR)"])
y_train = train_data["Revenue(INR)"]

X_test = test_data.drop(columns=["Revenue(INR)"])
y_test = test_data["Revenue(INR)"]

print("X_train shape: ", X_train.shape)
print("y_train shape: ", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape:  (1356, 2460)
y_train shape:  (1356,)
X_test shape: (340, 2460)
y_test shape: (340,)


In [14]:
print("Training feature data type: ")
print(X_train.dtypes)

Training feature data type: 
Number_of_Screens                    float64
Budget(INR)                          float64
Release_Period_Normal                float64
Whether_Remake_Yes                   float64
Whether_Franchise_Yes                float64
                                      ...   
Music_Director_Yuvan Shankar Raja    float64
Music_Director_Zakir Hussain         float64
Music_Director_Zubeen Garg           float64
Music_Director_Zubin Balaporia       float64
Music_Director_Zubunnisa Bangash     float64
Length: 2460, dtype: object


In [15]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("elastic_net", ElasticNet(
        alpha=0.1,
        l1_ratio=0.5,
        max_iter=10000,
        random_state=42
    )),
])

print("Elastic Net Model created successfully")

Elastic Net Model created successfully


In [16]:
model.fit(X_train, y_train)
print("Elastic Net Model trained successfully")

Elastic Net Model trained successfully


In [17]:
y_pred = model.predict(X_test)
print("Predictions generated successfully")
print("Number of predictions: ", len(y_pred))

Predictions generated successfully
Number of predictions:  340


In [20]:
prediction_data  = pd.DataFrame({
    "Actual Revenue": y_test.values,
    "Predicted Revenue": y_pred
})

prediction_data .head(10)

,Actual Revenue,Predicted Revenue
0,230000000.0,3.125818e+08
1,350000000.0,3.413746e+08
2,80000000.0,-3.377649e+07
3,160000000.0,1.000698e+08
4,20000000.0,5.067319e+07
5,70000000.0,-2.303645e+07
6,3000000.0,7.167496e+06
7,20000000.0,4.741470e+07
8,190000000.0,4.400744e+08
9,80000000.0,4.895662e+07


In [21]:
prediction_data["Predicted Revenue"] = prediction_data["Predicted Revenue"].round(2)

prediction_data.head(10)

,Actual Revenue,Predicted Revenue
0,230000000.0,3.125818e+08
1,350000000.0,3.413746e+08
2,80000000.0,-3.377649e+07
3,160000000.0,1.000698e+08
4,20000000.0,5.067319e+07
5,70000000.0,-2.303645e+07
6,3000000.0,7.167496e+06
7,20000000.0,4.741470e+07
8,190000000.0,4.400744e+08
9,80000000.0,4.895662e+07


In [22]:
print("Minimum predicted revenue:", y_pred.min())
print("Maximum predicted revenue:", y_pred.max())

Minimum predicted revenue: -106943648.25732061
Maximum predicted revenue: 1115862642.7508714


In [23]:
elastic_net_model = model.named_steps["elastic_net"]
print("Number of coefficients: ", len(elastic_net_model.coef_))
print("Intercept: ", elastic_net_model.intercept_)

Number of coefficients:  2460
Intercept:  150627747.0501475


In [24]:
feature_names = X_train.columns

coefficient_data = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": elastic_net_model.coef_
})

coefficient_data["Absolute_Coefficient"] = (
    coefficient_data["Coefficient"].abs()
)

coefficient_data = coefficient_data.sort_values(
    by="Absolute_Coefficient",
    ascending=False
)

coefficient_data.head(20)

,Feature,Coefficient,Absolute_Coefficient
0,Number_of_Screens,1.306699e+08,1.306699e+08
1,Budget(INR),4.503305e+07,4.503305e+07
625,Lead_Star_Shahrukh Khan,2.269877e+07,2.269877e+07
1523,Director_S.S. Rajamouli,1.978486e+07,1.978486e+07
448,Lead_Star_Prabhas,1.978486e+07,1.978486e+07
582,Lead_Star_Salman Khan,1.601969e+07,1.601969e+07
1973,Music_Director_DJ Chetas,1.488010e+07,1.488010e+07
256,Lead_Star_Hrithik Roshan,1.411593e+07,1.411593e+07
25,Lead_Star_Aamir Khan,1.408029e+07,1.408029e+07
848,Director_Ali Abbas Zafar,1.401792e+07,1.401792e+07


In [25]:
zero_coefficients = (elastic_net_model.coef_ == 0).sum()

print("Total features:", len(elastic_net_model.coef_))
print("Zero coefficients:", zero_coefficients)
print(
    "Non-zero coefficients:",
    len(elastic_net_model.coef_) - zero_coefficients
)

Total features: 2460
Zero coefficients: 356
Non-zero coefficients: 2104


In [26]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

model_path = "../models/elastic_net_model.pkl"

joblib.dump(model, model_path)

print("Model saved successfully")
print("Model path:", model_path)

Model saved successfully
Model path: ../models/elastic_net_model.pkl


In [27]:
loaded_model = joblib.load(model_path)

loaded_predictions = loaded_model.predict(X_test)

print("Saved model loaded successfully")
print("Predictions generated using saved model")

print(loaded_model[:5])

Saved model loaded successfully
Predictions generated using saved model
Pipeline(steps=[('scaler', StandardScaler()),
                ('elastic_net',
                 ElasticNet(alpha=0.1, max_iter=10000, random_state=42))])


In [28]:
comparision = pd.DataFrame({
    "Actual Revenue": y_test.values[:10],
    "Original Prediction": y_pred[:10],
    "Loaded Model prediction": loaded_predictions[:10]
})

print(comparision)

   Actual Revenue  Original Prediction  Loaded Model prediction
0     230000000.0         3.125818e+08             3.125818e+08
1     350000000.0         3.413746e+08             3.413746e+08
2      80000000.0        -3.377649e+07            -3.377649e+07
3     160000000.0         1.000698e+08             1.000698e+08
4      20000000.0         5.067319e+07             5.067319e+07
5      70000000.0        -2.303645e+07            -2.303645e+07
6       3000000.0         7.167496e+06             7.167496e+06
7      20000000.0         4.741470e+07             4.741470e+07
8     190000000.0         4.400744e+08             4.400744e+08
9      80000000.0         4.895662e+07             4.895662e+07


In [29]:
print("====================================")
print("MODEL BUILDING COMPLETED")
print("====================================")

print("Model: Elastic Net Regression")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Number of features:", X_train.shape[1])
print("Alpha:", elastic_net_model.alpha)
print("L1 Ratio:", elastic_net_model.l1_ratio)
print("Model saved at:", model_path)

MODEL BUILDING COMPLETED
Model: Elastic Net Regression
Training samples: 1356
Testing samples: 340
Number of features: 2460
Alpha: 0.1
L1 Ratio: 0.5
Model saved at: ../models/elastic_net_model.pkl
